In [0]:
display(dbutils.fs.ls("/Volumes/databricks_project1/bronze/landing_volume"))

In [0]:
%pip install openpyxl

In [0]:
%restart_python

In [0]:
CATALOG = "databricks_project1"
SCHEMA = "bronze"

VOLUME_PATH = "/Volumes/databricks_project1/bronze/landing_volume"

EMPLOYEE_FILE = f"{VOLUME_PATH}/Employee_Payroll.xlsx"
LABOR_FILE = f"{VOLUME_PATH}/Labor_Position.xlsx"

### -- Employee_Payroll.xlsx Ingestion --


 --Read the employee_file--

In [0]:
employee_df = (
    spark.read
    .format("excel")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(EMPLOYEE_FILE)
)

display(employee_df)

In [0]:
employee_df.printSchema()

In [0]:
employee_df.columns

--Create Data frame--



In [0]:
import pandas as pd

pdf = pd.read_excel(EMPLOYEE_FILE)
pdf = pdf.astype(str)

employee_df = spark.createDataFrame(pdf)



In [0]:
employee_df.columns
display(employee_df)

In [0]:
pdf["DOB"].head(10)

In [0]:
from pyspark.sql.functions import when, col

for column in employee_df.columns:
    employee_df = employee_df.withColumn(
        column,
        when(
            (col(column) == "nan") |
            (col(column) == "NaN") |
            (col(column) == "None") |
            (col(column) == ""),
            None
        ).otherwise(col(column))
    )

display(employee_df)

--data validation --

In [0]:
print(f"Employee Record Count: {employee_df.count()}")

In [0]:
employee_df.printSchema()

--Add Audit Columns to Employee--

In [0]:
from pyspark.sql.functions import current_timestamp, lit

employee_df = (
    employee_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit("Employee_Payroll.xlsx"))
)

In [0]:
display(employee_df)

--Save as Bronze Delta Table--

In [0]:
employee_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("databricks_project1.bronze.employee_payroll")

In [0]:
spark.sql("SHOW TABLES IN databricks_project1.bronze").show()

In [0]:
display(
    spark.table("databricks_project1.bronze.employee_payroll")
)

###  -- Labor_Position.xlsx Ingestion --

--Read The Labor_File --

In [0]:
CATALOG = "databricks_project1"
SCHEMA = "bronze"

VOLUME_PATH = "/Volumes/databricks_project1/bronze/landing_volume"
LABOR_FILE = f"{VOLUME_PATH}/Labor_Position.xlsx"

In [0]:
%pip install openpyxl

In [0]:
import pandas as pd

labor_pdf = pd.read_excel(LABOR_FILE)

labor_pdf.head()

In [0]:
labor_pdf = labor_pdf.astype(str)

In [0]:
labor_df = spark.createDataFrame(labor_pdf)

display(labor_df)

In [0]:
from pyspark.sql.functions import when, col

for column in labor_df.columns:
    labor_df = labor_df.withColumn(
        column,
        when(
            (col(column) == "nan") |
            (col(column) == "NaN") |
            (col(column) == "None") |
            (col(column) == ""),
            None
        ).otherwise(col(column))
    )

display(labor_df)

In [0]:
print(f"Labor Position Record Count: {labor_df.count()}")

In [0]:
labor_df.printSchema()

In [0]:
from pyspark.sql.functions import col

labor_df.groupBy("Labor_Position_Code") \
        .count() \
        .filter(col("count") > 1) \
        .show()

In [0]:
from pyspark.sql.functions import sum, when, col

labor_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in labor_df.columns
]).show()

--Add Audit Columns--

In [0]:
from pyspark.sql.functions import current_timestamp, lit

labor_df = (
    labor_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit("Labor_Position.xlsx"))
)

display(labor_df)

-- Save as delta table --



In [0]:
labor_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("databricks_project1.bronze.labor_position")

In [0]:
display(
    spark.table("databricks_project1.bronze.employee_payroll")
)